In [3]:
import os
from tqdm import tqdm
import sys
if not hasattr(sys.modules[__name__], "cwd_changed"):
    os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(__name__))))
    sys.modules[__name__].cwd_changed = True

import warnings 
warnings.filterwarnings("ignore")
import logging
logging.getLogger('pgmpy').setLevel(logging.WARNING)


import pandas as pd
from utils.graph import dag_to_cpdag
# from metrics.graph import compare_dags
from utils.results import *
from analyses.helpers import load_results


In [88]:

from __future__ import annotations

import os
import hashlib
from typing import Any, Dict, Iterable, List, Optional, Tuple
import pandas as pd
from synthetic.structure import *
from metrics.synergy import find_synergistic_triplets
from synthetic.binary import *
from idtxl.data import Data
from idtxl.bivariate_pid import BivariatePID

def broja_synergy(df, source1, source2, target):

    s1, _ = pd.factorize(df[source1], sort=True)
    s2, _ = pd.factorize(df[source2], sort=True)
    t,  _ = pd.factorize(df[target], sort=True)

    data = Data(np.vstack((s1, s2, t)), 'ps', normalise=False)

    settings = {
        'alpha': 0.1,
        'alph_s1': int(s1.max() + 1),
        'alph_s2': int(s2.max() + 1),
        'alph_t': int(t.max() + 1),
        'max_unsuc_swaps_row_parm': 60,
        'num_reps': 63,
        'max_iters': 1000,
        'pid_estimator': 'TartuPID',
        'verbose': False,
        'lags_pid': [0, 0],   # or remove if you do not want lagged PID
    }

    pid_analysis = BivariatePID()
    results = pid_analysis.analyse_single_target(
        settings=settings,
        data=data,
        target=2,
        sources=[0, 1],
    )
    return results.get_single_target(2)['syn_s1_s2']

def evaluate_triplet(df, triplet):
    highest_pid = 0

    triplet = list(map(str, triplet))

    for col in triplet:
        target = col
        source1, source2 = [c for c in triplet if c != target]
        pid = broja_synergy(df, source1, source2, target)
        # print(f"Evaluating triplet {source1}, {source2} -> {target}: PID = {pid}")

        if pid > highest_pid:
            highest_pid = pid
            final_triplet = (source1, source2, target)

    # print(final_triplet)
    # print(f"\nHighest PID for triplet {final_triplet}: {highest_pid} with target {final_triplet[2]}\n")
    return (final_triplet, highest_pid)


In [111]:
n_nodes = 50
n_samples = 10_000
fam = "ER"
dag, topo, info = sample_dag_family_with_caps(
n_nodes=n_nodes,
family=fam,
# seed=structure_seed,
max_indeg=int(2),
max_outdeg=int(2),
)

p_syn=1
sharpness=10

node_cpd, cpd_info = build_node_cpds_from_dag(
    dag,
    topo=topo,
    # seed_params=seed_params,
    # root_p=float(root_p),
    # p_copy_flip=float(p_copy_flip),
    p_syn=float(p_syn),
    sharpness=float(sharpness),
)
df, dag_str, node_spec, meta = simulate_from_dag_cpds(
                            dag,
                            node_cpd,
                            n_samples=n_samples,
                            p_flip_global=0,
                            topo=topo,
                        )
                   

In [112]:
ii_triplets = find_synergistic_triplets(df)

In [114]:
from joblib import Parallel, delayed
results = Parallel(n_jobs=10, backend="loky")(
            delayed(evaluate_triplet)(df, triplet) for triplet in ii_triplets
        )
    
count = 0
for i in results:
    triplet_i = i[0]
    pid_i = i[1]
    # print(f"Triplet: {i[0]} with PID {i[1]}")
    if int(triplet_i[2]) > int(triplet_i[0]) and int(triplet_i[2]) > int(triplet_i[1]):
        count += 1

ratio = count / len(results)
print(f"Ratio of triplets where target has higher PID than sources: {ratio:.2f}")

Ratio of triplets where target has higher PID than sources: 0.36


## Lower Sharpness

In [115]:
n_nodes = 20
n_samples = 10_000
p_syn=1
sharpness=1

node_cpd, cpd_info = build_node_cpds_from_dag(
    dag,
    topo=topo,
    # seed_params=seed_params,
    # root_p=float(root_p),
    # p_copy_flip=float(p_copy_flip),
    p_syn=float(p_syn),
    sharpness=float(sharpness),
)
df, dag_str, node_spec, meta = simulate_from_dag_cpds(
                            dag,
                            node_cpd,
                            n_samples=n_samples,
                            p_flip_global=0,
                            topo=topo,
                        )
                   

In [116]:
ii_triplets = find_synergistic_triplets(df)

In [117]:
from joblib import Parallel, delayed
results = Parallel(n_jobs=10, backend="loky")(
            delayed(evaluate_triplet)(df, triplet) for triplet in ii_triplets
        )
    
count = 0
for i in results:
    triplet_i = i[0]
    pid_i = i[1]
    # print(f"Triplet: {i[0]} with PID {i[1]}")
    if int(triplet_i[2]) > int(triplet_i[0]) and int(triplet_i[2]) > int(triplet_i[1]):
        count += 1

ratio = count / len(results)
print(f"Ratio of triplets where target has higher PID than sources: {ratio:.2f}")

Ratio of triplets where target has higher PID than sources: 0.34


In [80]:
## Bit-flip Noise

In [118]:

df, dag_str, node_spec, meta = simulate_from_dag_cpds(
                            dag,
                            node_cpd,
                            n_samples=n_samples,
                            p_flip_global=0.3,
                            topo=topo,
                        )
ii_triplets = find_synergistic_triplets(df)

In [119]:
from joblib import Parallel, delayed
results = Parallel(n_jobs=10, backend="loky")(
            delayed(evaluate_triplet)(df, triplet) for triplet in ii_triplets
        )
    
count = 0
for i in results:
    triplet_i = i[0]
    pid_i = i[1]
    # print(f"Triplet: {i[0]} with PID {i[1]}")
    if int(triplet_i[2]) > int(triplet_i[0]) and int(triplet_i[2]) > int(triplet_i[1]):
        count += 1

ratio = count / len(results)
print(f"Ratio of triplets where target has higher PID than sources: {ratio:.2f}")

Ratio of triplets where target has higher PID than sources: 0.24
